In [0]:
catalog = "sandbox"
schema = "bronze"
df_equipos_bronze = spark.table(f"{catalog}.{schema}.equipos")

In [0]:
display(df_equipos_bronze.limit(50))

In [0]:
df_equipos_clean = df_equipos_bronze.fillna(
    "Desconocido", 
    subset=["fabricante", "region", "estado"]
    )

In [0]:
from pyspark.sql.functions import upper, trim, regexp_replace, col

df_equipos_clean = df_equipos_clean.withColumn(
    "region", 
    upper(trim(
            regexp_replace(col("region"), "\\.", "")
            ))
    )

In [0]:
from pyspark.sql.functions import translate

df_equipos_clean = df_equipos_clean.withColumn(
    "region",
    translate(col("region"), "ÁÉÍÓÚÜ", "AEIOUU")
    )

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

estados_mapping = {
    "operativo": "operativo",
    "op": "operativo",
    "en mantenimiento": "en mantenimiento",
    "mant": "en mantenimiento",
    "en_mantenimiento": "en mantenimiento",
    "fds": "fuera de servicio",
    "fuera_de_servicio": "fuera de servicio",
    "fuera de servicio": "fuera de servicio"
}

def normalizar_estado(estado):
    estado = estado.lower().strip()
    return estados_mapping.get(estado, "desconocido")

normalizar_estado_udf = udf(normalizar_estado, StringType())
df_equipos_clean = df_equipos_clean.withColumn(
    "estado",
    normalizar_estado_udf(col("estado"))
)

In [0]:
from pyspark.sql.functions import try_to_timestamp, coalesce, lit

formatos = [
    "yyyy-MM-dd HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss",
    "dd/MM/yyyy HH:mm",
    "dd-MM-yyyy",
    "yyyyMMddHHmmss",
    "yyyyMMdd"
]

df_equipos_clean = (
    df_equipos_clean.withColumn(
    "fecha_instalacion",
    coalesce(*[try_to_timestamp(col("fecha_instalacion"), lit(fmt)) for fmt in formatos])
    )
)

In [0]:
df_equipos_clean.createOrReplaceTempView("equipos_clean_vw")

In [0]:
%sql
select * from equipos_clean_vw

In [0]:
%sql
MERGE INTO sandbox.silver.equipos tgt
USING equipos_clean_vw src
ON tgt.equipo_id = src.equipo_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *